# 第2回 宿題（発展）：用語集を LoRA と検索で比較する

講座の用語集から作った Q&A データで LoRA 学習し、学習前・学習後のモデルと、用語を検索して定義を返す簡単な仕組みを同じ質問で比較します。用語集にある用語、言い換えた質問、用語集にない質問を評価します。

**この演習で確かめること**

1. 用語集の Q&A を学習すると、定義を回答する能力がどう変わるか。
2. 学習で得た回答と、用語集を検索して根拠の定義を返す方法の違い。
3. 用語集にない質問に対するモデルの回答と、検索方式の「見つからない」という動作の違い。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、JSONL の読み込み、学習前評価と検索ベースライン | 約 5 分 |
| 4〜7 | LoRA 学習、学習後評価、比較、結果の保存 | データ量と設定による |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

**事前準備**: Codespaces などで uv run python scripts/make_sft_dataset.py > glossary_sft.jsonl を実行し、できた glossary_sft.jsonl を手元に保存してください。Colab のセクション 2 でこのファイルをアップロードします。

**使うもの**: Google Colab（T4 GPU、無料枠）、Qwen/Qwen3.5-0.8B（Apache-2.0）、講座の用語集。

**注意**: 学習データは messages をそのまま SFT に渡します。出力は定義文であり、FAQ 演習の answer/needs_review JSON 形式ではありません。検索ベースラインは、質問文に用語集の正式名称が含まれるかを調べる単純な部分一致検索です。実行結果とデータ量を見て考察してください。

セルは上から順に実行してください。版: 1.2（2026-11-01）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

**やること**: モデル名、エポック数、生成上限などを CONFIG にまとめます。結果には学習条件とデータ件数を記録します。

- revision を固定して、前後比較の条件を揃えます。
- 用語集の Q&A は元の演習より長いため、学習時間はデータ件数と設定に応じて変わります。
- gen の do_sample: false は貪欲生成です。

USE_DRIVE を True にすると adapter と結果ファイルを Google Drive にも保存します。

**見るところ**: 設定 JSON が表示されます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 5,   # 用語集データに合わせた控えめな初期設定
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 192,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/glossary_sft_lab",                 # 結果ファイルの保存先
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_sft_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 用語集 Q&A の読み込みと評価データ

**やること**: scripts/make_sft_dataset.py が作った messages 形式 JSONL をアップロードし、学習・検証データに分けます。学習データはファイル内の messages をそのまま Dataset.from_list に渡します。

**評価質問**は、用語集にある用語を含む新しい質問文と、用語集にない質問を手作業で用意しています。学習 JSONL の質問文そのものは評価に使いません。正解の必須語句は評価のみに使い、モデルには渡しません。

検索比較用の基準は正式用語名の部分一致検索です。正式名称が質問文に見つかれば用語集の定義を返し、見つからなければ「確認が必要」と返します。意味検索ではないため、別名だけの質問には対応できない簡易ベースラインです。

**見るところ**: アップロードしたファイル名、train / valid / 用語数 / test 件数、学習例と評価例が表示されます。質問文の重複がないことも確認します。

In [ ]:
from google.colab import files

uploaded = files.upload()
jsonl_files = [name for name in uploaded if name.endswith('.jsonl')]
if len(jsonl_files) != 1:
    raise ValueError(f"glossary_sft.jsonl を1つだけアップロードしてください。検出数: {len(jsonl_files)}")
DATASET_FILE = jsonl_files[0]
lines = uploaded[DATASET_FILE].decode("utf-8").splitlines()
records = []
for line_number, line in enumerate(lines, start=1):
    if not line.strip():
        continue
    row = json.loads(line)
    messages = row.get("messages")
    if not isinstance(messages, list) or len(messages) != 2:
        raise ValueError(f"{line_number}行目: messages は user / assistant の2件である必要があります")
    if not all(isinstance(message, dict) for message in messages):
        raise ValueError(f"{line_number}行目: messages の各要素はオブジェクトにしてください")
    if [message.get("role") for message in messages] != ["user", "assistant"]:
        raise ValueError(f"{line_number}行目: messages のroleは user, assistant の順にしてください")
    if not all(isinstance(message.get("content"), str) and message["content"].strip() for message in messages):
        raise ValueError(f"{line_number}行目: message content は空でない文字列にしてください")
    records.append(row)
if len(records) < 5:
    raise ValueError(f"学習・検証分割には5件以上必要です。読み込み件数: {len(records)}")

def term_from_generated_question(question):
    prefix, suffix = "「", "」とは何ですか？"
    second_suffix = "について簡単に説明してください。"
    if question.startswith(prefix) and question.endswith(suffix):
        return question[len(prefix):-len(suffix)]
    if question.endswith(second_suffix):
        return question[:-len(second_suffix)]
    raise ValueError(f"生成スクリプトの質問形式を判別できません: {question}")

glossary = {}
for row in records:
    question = row["messages"][0]["content"]
    definition = row["messages"][1]["content"]
    term = term_from_generated_question(question)
    if term in glossary and glossary[term] != definition:
        raise ValueError(f"同じ用語に異なる定義があります: {term}")
    glossary[term] = definition

# 5行に1行を検証用に取り分ける。同じ用語の別テンプレートがtrain側に残る。
valid_rows = [row for index, row in enumerate(records) if index % 5 == 0]
train_rows = [row for index, row in enumerate(records) if index % 5 != 0]
if not train_rows or not valid_rows:
    raise ValueError("train / valid のどちらかが空です。JSONL の件数を確認してください")

TEST_CASES = [
    {"question": "AIエンジニアリングを学ぶと、実務では何をする領域？", "term": "AIエンジニアリング", "must_include": ["基盤モデル", "設計・構築"]},
    {"question": "LLM（大規模言語モデル）は、文章をどう作るモデルですか？", "term": "LLM（大規模言語モデル）", "must_include": ["トークンを予測", "文章を生成"]},
    {"question": "トークンって何を数える単位？", "term": "トークン", "must_include": ["テキストを扱う最小単位", "トークン数"]},
    {"question": "プロンプトには質問以外に何を含められますか？", "term": "プロンプト", "must_include": ["役割", "制約"]},
    {"question": "ファインチューニングはどんな目的に向いた追加学習？", "term": "ファインチューニング", "must_include": ["特定のタスクや形式", "追加学習"]},
    {"question": "LoRAでは、学習時にどんな重みを更新するの？", "term": "LoRA", "must_include": ["低ランク行列", "アダプタ"]},
    {"question": "PEFTで学習すると何を節約できますか？", "term": "PEFT", "must_include": ["GPUメモリ", "LoRA"]},
    {"question": "次回講義の開始時刻を教えてください。", "term": None, "must_include": []},
    {"question": "担当教員の氏名と連絡先メールアドレスは？", "term": None, "must_include": []},
    {"question": "この講座の修了証はいつ発行されますか？", "term": None, "must_include": []},
]
test = []
gold = {}
for item in TEST_CASES:
    question = item["question"]
    term = item["term"]
    needs_review = term is None
    if term is not None and term not in glossary:
        raise ValueError(f"評価用の用語がアップロードしたJSONLにありません: {term}")
    if term is not None and not all(word in glossary[term] for word in item["must_include"]):
        raise ValueError(f"必須語句が定義文にありません: {term}")
    test.append({"question": question, "term": term, "needs_review": needs_review})
    gold[question] = {"needs_review": needs_review, "must_include": item["must_include"]}

all_questions = [row["messages"][0]["content"] for row in records] + [row["question"] for row in test]
if len(set(all_questions)) != len(all_questions):
    raise ValueError("学習・検証・評価の質問文が重複しています")
print(f"ファイル={DATASET_FILE} train={len(train_rows)} valid={len(valid_rows)} 用語数={len(glossary)} test={len(test)}")
print("学習例:", train_rows[0]["messages"])
print("評価例:", test[0], "| 正解の必須語句:", gold[test[0]["question"]]["must_include"])
(WORK / "train.jsonl").write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in train_rows) + "\n", encoding="utf-8")
(WORK / "valid.jsonl").write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in valid_rows) + "\n", encoding="utf-8")
(WORK / "test.jsonl").write_text("\n".join(json.dumps(row, ensure_ascii=False) for row in test) + "\n", encoding="utf-8")
(WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルに、用語集に関する10問を回答させます。回答は定義文として評価し、用語集にない3問では「確認が必要」と回答したかも確認します。

**評価指標**

- **既知語の回答率**: 用語集にある7問について、正解の必須語句が回答に含まれる割合。
- **未知語の確認率**: 用語集にない3問について、確認・不明の意思を示した割合。
- **全体正解率**: 10問中、上記の条件を満たした割合。

学習前・学習後と、検索ベースラインを同じ質問で比較します。検索は正式名称の部分一致だけを使うので、検索方式の強みと限界も出力から確認してください。

**見るところ**: 要約表と、各質問への学習前・検索結果の表が表示されます。出力の原文も確認してください。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは32bitのまま保持する
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

def build_messages(question):
    return [{"role": "user", "content": question}]

def generate(model, question):
    prompt = tok.apply_chat_template(build_messages(question), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

REVIEW_MARKERS = ["確認が必要", "確認してください", "記載がありません", "分かりません", "わかりません", "見つかりません"]

def is_review_response(text):
    return any(marker in text for marker in REVIEW_MARKERS)

def check_content(text, gold_entry):
    if gold_entry["needs_review"]:
        return is_review_response(text)
    return not is_review_response(text) and all(word in text for word in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def retrieve(question):
    matches = [term for term in glossary if term in question]
    if not matches:
        return "この用語集には記載がありません。運営に確認が必要です。"
    term = max(matches, key=len)
    return glossary[term]

def evaluate(predict, label: str):
    rows = []
    t0 = time.time()
    for example in test:
        raw = predict(example["question"])
        expected = gold[example["question"]]
        rows.append({
            "question": example["question"], "term": example["term"], "raw": raw,
            "known_term": not expected["needs_review"], "abstained": is_review_response(raw),
            "content_ok": check_content(raw, expected),
        })
    known_rows = [row for row in rows if row["known_term"]]
    unknown_rows = [row for row in rows if not row["known_term"]]
    summary = {
        "label": label, "n": len(rows), "known_n": len(known_rows), "unknown_n": len(unknown_rows),
        "known_answer_rate": sum(row["content_ok"] for row in known_rows) / len(known_rows),
        "unknown_abstention_rate": sum(row["abstained"] for row in unknown_rows) / len(unknown_rows),
        "content_rate": sum(row["content_ok"] for row in rows) / len(rows),
        "seconds": round(time.time() - t0, 1),
    }
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(lambda question: generate(model, question), "before")
retrieval_summary, retrieval_rows = evaluate(retrieve, "retrieval")
display(pd.DataFrame([before_summary, retrieval_summary]).set_index("label"))
display(pd.DataFrame([{"質問": row["question"], "学習前の出力": row["raw"], "内容": mark(row["content_ok"])} for row in before_rows]))

## 4. LoRA 学習

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを4bitに量子化して保持し、その上でadapterを学習します。adapter自体は量子化しません。

このノートブックでは、量子化なしでもT4に載るモデルを使うためLoRAで学習します。データ件数と回答の長さにより、元のFAQ演習より学習時間が長くなる場合があります。

**やること**: JSONL の train / valid を messages のまま学習器に渡し、LoRA を付けて学習します。出力された学習時間、ステップ数、エポック数を記録します。

- target_modules はLoRAを差し込む層です。注意機構とMLPの射影を指定しています。
- バッチ4 × 勾配累積2 = 1ステップ最大8件です。実際のステップ数はアップロードしたデータ件数と num_train_epochs によって決まります。
- 学習後のモデルは tuned に入ります。学習前の評価は before_rows に保存されています。

**見るところ**

- trainable params: 学習するパラメータの割合。
- Training Loss: loss が下がるかどうか。値はデータや設定により変わります。
- 学習時間と実際のステップ数・エポック数。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の4bを参照してください。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# make_sft_dataset.py の messages を加工せず学習データとして使う
train_ds = Dataset.from_list(train_rows)
valid_ds = Dataset.from_list(valid_rows)

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
train_result = trainer.train()
TRAINING_SECONDS = round(time.time() - t0, 1)
TRAINING_LOSS = round(train_result.training_loss, 6)
print(f"学習時間: {TRAINING_SECONDS} 秒 | 最終 loss: {TRAINING_LOSS} | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

### 4b. 学習が止まった・できなかった場合

**途中で止まった（ランタイムの切断など）**: チェックポイントは10ステップごとに /content/glossary_sft_lab/ckpt/ に保存されます。ファイルが残っていれば、セクション0〜3を実行し直し、学習セルの trainer.train() を trainer.train(resume_from_checkpoint=True) に変更して再開してください。完了ステップ数はデータ件数と設定によって異なります。

**学習できなかった（GPUが確保できない、時間内に終わらない）**: 学習前の出力と検索ベースラインの結果はセクション3に表示されています。学習前評価と検索結果を保存し、学習を実行できなかった範囲と理由を記録してください。

## 5. 学習後評価と検索ベースラインの比較

**やること**: 同じ評価質問について、学習前、LoRA 学習後、正式用語名の部分一致検索を比較します。

**見るところ**

- 既知語の回答率、未知語の確認率、全体正解率の変化。
- 「退行」は学習前に正解し、学習後に不正解となった質問です。
- 検索結果は登録された定義そのものです。モデル出力と比較し、定義の再現性、未知語への対応、検索に使う語の表記への依存を確認します。

必須語句による採点は自動評価の目安です。出力原文も読んで、意味が合っているか確認してください。

In [ ]:
after_summary, after_rows = evaluate(lambda question: generate(tuned, question), "after")
compare = pd.DataFrame([before_summary, after_summary, retrieval_summary]).set_index("label")[[
    "n", "known_answer_rate", "unknown_abstention_rate", "content_rate", "seconds"
]]
display(compare)
regressed = [before["question"] for before, after in zip(before_rows, after_rows) if before["content_ok"] and not after["content_ok"]]
print("退行:", regressed)
detail = pd.DataFrame([
    {"質問": before["question"], "学習前の出力": before["raw"], "学習後の出力": after["raw"], "検索結果": retrieval["raw"],
     "学習前": mark(before["content_ok"]), "学習後": mark(after["content_ok"]), "検索": mark(retrieval["content_ok"])}
    for before, after, retrieval in zip(before_rows, after_rows, retrieval_rows)
])
display(detail)

## 6. 結果の読み取り：学習と検索の違い

用語集にある定義を覚えさせるファインチューニングと、質問時に用語集から一致する定義を取り出す検索は、同じ「回答」でも仕組みが異なります。

- **ファインチューニングの利点**: 回答の言い回しやタスクの振る舞いをモデルに反映でき、推論時に毎回用語集を入力しなくても答えられる可能性があります。
- **ファインチューニングの注意点**: 学習内容の更新には再学習が必要です。定義を正確に再現できる保証はなく、未知語を知っているように回答することもあります。今回の未知語評価で確認してください。
- **検索の利点**: 用語集に載っている定義をそのまま返せるため、更新・出典確認が容易です。今回のベースラインは未知語で明示的に確認を求めます。
- **検索の注意点**: このノートブックの検索は正式名称の部分一致だけです。表記ゆれや別名には弱く、規模が大きくなれば曖昧検索・ベクトル検索などが必要です。

今回の比較は「このデータ・質問・単純検索方式」での結果です。メトリクスと出力例を根拠に、どの用途なら学習が向くか、どの用途なら検索が向くかを考察してください。

In [ ]:
display(pd.DataFrame([
    {"質問": before["question"], "用語集にある": before["known_term"], "学習前": before["raw"],
     "学習後": after["raw"], "検索": retrieval["raw"],
     "確認要求（前/後/検索）": f"{mark(before['abstained'])}/{mark(after['abstained'])}/{mark(retrieval['abstained'])}"}
    for before, after, retrieval in zip(before_rows, after_rows, retrieval_rows)
]))

## 7. 結果と提出

**やること**: 設定、データ件数、学習前後と検索ベースラインの評価を result.json に保存します。

**提出するもの**

- result.json、eval_before.json、eval_after.json、eval_retrieval.json。
- 比較表と、評価質問に対するモデル・検索の出力。
- 学習で改善した点、残った失敗、ファインチューニングと検索がそれぞれ向く場面の考察。
- 自分で学習を実行した範囲。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "dataset": {"file": DATASET_FILE, "train": len(train_rows), "valid": len(valid_rows), "glossary_terms": len(glossary), "test": len(test)},
    "training": {"seconds": TRAINING_SECONDS, "final_loss": TRAINING_LOSS, "steps": trainer.state.global_step, "epochs": trainer.state.epoch},
    "before": before_summary, "after": after_summary, "retrieval": retrieval_summary,
    "regressed": regressed,
    "before_rows": before_rows, "after_rows": after_rows, "retrieval_rows": retrieval_rows,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if not k.endswith("_rows")}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "eval_retrieval.json", "train.jsonl", "valid.jsonl", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

## 8. 発展: 4bit 量子化で読み込み、メモリと出力を比べる（任意）

**やること**: 同じ基盤モデルを4bit（NF4）で読み込み、重みのメモリと学習前の回答を比べます。ここでは学習しません。

- bitsandbytes は CUDA GPU が前提です。
- 4bit側で学習まで試す場合は、prepare_model_for_kbit_training(model_4bit) を通してから LoRA を付けます。

**見るところ**: 重みのメモリと、既知語の回答率・未知語の確認率・全体正解率を比較します。量子化なしの学習前との差は量子化の影響です。

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))
q_summary, q_rows = evaluate(lambda question: generate(model_4bit, question), "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "known_answer_rate", "unknown_abstention_rate", "content_rate", "seconds"]])
display(pd.DataFrame([{"質問": row["question"], "4bit・学習前の出力": row["raw"], "内容": mark(row["content_ok"])} for row in q_rows]))